In [1]:
!pip install arxiv

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB 653.6 kB/s eta 0:00:07
   - -------------------------------------- 0.1/4.0 MB 1.2 MB/s eta 0:00:04
   ----- ---------------------------------- 0.6/4.0 MB 3.9 MB/s eta 0:00:01
   -------------- ------------------------- 1.5/4.0 MB 7.9 MB/s eta 0:00:01
   ------------------ --------------------- 1.8/4.0 MB 8.9 MB/s eta 0:00:01
   --------------------- ------------------ 2.2/4.0 MB 8.7 MB/s eta 0:00:01
   ----------------------------------- ---- 3.5/4.0 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 11.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/64.9 kB ? eta -:--:--
   ---------------------------------------- 64.9/64.9 kB 3.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


In [15]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)

In [16]:
wiki.name

'wikipedia'

In [ ]:
##kinda custom tool
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,HuggingFaceEmbeddings())
retriever=vectordb.as_retriever()
retriever

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3064.80it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019D7982E610>, search_kwargs={})

In [12]:
from langchain_core.tools import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith_search",
                      "Search for information about LnagSmith. For any questions about LangSmith you must use this tool.")


In [13]:
retriever_tool.name

'langsmith_search'

In [14]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [17]:
tools=[wiki,arxiv,retriever_tool]

In [18]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\langchainrag1\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200)),
 StructuredTool(name='langsmith_search', description='Search for information about LnagSmith. For any questions about LangSmith you must use this tool.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000019E45CB36A0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000019E45CB3E20>)]

In [21]:
docs = retriever.invoke("What is LangSmith?")
print(docs[:2])

[Document(id='7520a3ca-e331-462e-9f84-c16aa8d9d8a2', metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'language': 'en'}, page_content='cause, and resolve them with LangSmith Engine.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.'), Document(id='48f87af6-e417-46d9-bce1-a10cbb3586ca', metadata={'source': 'https://docs.smith.langchain.com/', 'title': 'LangSmith Observability - Docs by LangChain', 'description': 'Instrument your LLM application, investigate traces, and monitor performance in production with LangSmith.', 'languag

In [20]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

llm = ChatOpenAI(
    api_key=os.getenv("SAMBANOVA_API_KEY"),
    base_url="https://api.sambanova.ai/v1",
    model="Meta-Llama-3.3-70B-Instruct"
)

response = llm.invoke("What is LangSmith?")
print(response.content)

I do not have access to a search engine to provide information on LangSmith.


In [29]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful AI assistant.

    Use:
    - langsmith_search for LangSmith questions
    - arxiv for research paper questions
    - wikipedia for general knowledge questions
    """
)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is Machine Learning?"
            }
        ]
    }
)

print(response["messages"][-1].content)

Machine learning is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generate predictions or decisions.


In [30]:
import langchain
print(langchain.__version__)

1.3.10
